<a href="https://colab.research.google.com/github/talhanoor23/algorithmic-trading/blob/main/Creating_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential

from datetime import datetime, timedelta
import requests

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# apikey = 'c892c9b4a3464b2689f46732c5bfbdc5'
# symbol = "BTC/USD"
# interval = "5min"

In [ ]:
# 8 request is allowed per minute so here time is not adjusted so we comment it and use next row.

# # ---- Set the full date range you want ----
# start_date_global = datetime(2020, 1, 1)
# end_date_global = datetime(2025, 7, 7)   # you can make this datetime.today()

# # 15-day per chunk
# chunk_days = 15

# # Calculate how many chunks needed
# total_days = (end_date_global - start_date_global).days
# n_chunks = total_days // chunk_days + 1

# df = pd.DataFrame()

# print(f"Total Chunks Needed: {n_chunks}")

# for i in range(n_chunks):
#     # Build chunk date range
#     chunk_end = end_date_global - timedelta(days=i * chunk_days)
#     chunk_start = chunk_end - timedelta(days=chunk_days)

#     # Stop when chunk_start goes earlier than 2020-01-01
#     if chunk_start < start_date_global:
#         chunk_start = start_date_global

#     url = (f"https://api.twelvedata.com/time_series?"
#            f"symbol={symbol}&interval={interval}"
#            f"&start_date={chunk_start.date()}"
#            f"&end_date={chunk_end.date()}"
#            f"&apikey={apikey}")

#     print(f"Downloading {chunk_start.date()} → {chunk_end.date()}")

#     data = requests.get(url).json()

#     if "values" in data:
#         temp = pd.DataFrame(data["values"])
#         temp['datetime'] = pd.to_datetime(temp['datetime'])

#         # Remove last row if not the final chunk
#         if i != (n_chunks - 1):
#             temp = temp[:-1]

#         df = pd.concat([df, temp], ignore_index=True)
#     else:
#         print("Error:", data.get("message"))

# # Final sort
# df = df.sort_values("datetime").reset_index(drop=True)

# print("Done! Total rows:", len(df))
# print(df.head())
# print(df.tail())

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import time
import os

# ------------------------ SETTINGS ------------------------
apikey = 'c892c9b4a3464b2689f46732c5bfbdc5'
symbol = "BTC/USD"
interval = "5min"
chunk_days = 15  # adjust chunk size as needed

start_date_global = datetime(2021, 5, 1)
end_date_global = datetime(2025, 11, 25)

output_csv = "btc_5min_dataset.csv"
checkpoint_file = "checkpoint.txt"
max_requests_per_minute = 8  # TwelveData free tier
# ----------------------------------------------------------

# ------------------------ LOAD EXISTING DATA ------------------------
if os.path.exists(output_csv):
    df = pd.read_csv(output_csv)
    df['datetime'] = pd.to_datetime(df['datetime'])
    print(f"Resuming: Loaded existing dataset with {len(df)} rows.")
else:
    df = pd.DataFrame()

# Load checkpoint or start fresh
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r") as f:
        last_downloaded_date = datetime.strptime(f.read().strip(), "%Y-%m-%d %H:%M:%S")
    print("Resuming from checkpoint:", last_downloaded_date)
elif not df.empty:
    last_downloaded_date = df['datetime'].max()
    print("Resuming from last datetime in CSV:", last_downloaded_date)
else:
    last_downloaded_date = start_date_global
    print("Starting fresh download from:", last_downloaded_date)

# ------------------------ MAIN DOWNLOAD LOOP ------------------------
current_start = last_downloaded_date
request_count = 0

while current_start < end_date_global:
    current_end = current_start + timedelta(days=chunk_days)
    if current_end > end_date_global:
        current_end = end_date_global

    # Skip chunk if fully exists
    if not df.empty and current_end <= df['datetime'].max():
        print(f"Skipping {current_start.date()} → {current_end.date()} (already downloaded)")
        current_start = current_end
        continue

    # Build API URL
    url = (
        f"https://api.twelvedata.com/time_series?"
        f"symbol={symbol}&interval={interval}"
        f"&start_date={current_start.date()}"
        f"&end_date={current_end.date()}"
        f"&apikey={apikey}"
    )

    # Rate-limit handling
    if request_count >= max_requests_per_minute:
        print("⏳ Waiting 60 seconds due to API limit...")
        time.sleep(60)
        request_count = 0

    # Retry logic
    for attempt in range(3):
        try:
            response = requests.get(url)
            data = response.json()
            if "values" not in data:
                raise Exception(data.get("message", "Unknown API error"))
            break
        except Exception as e:
            print(f"Request failed (attempt {attempt+1}/3): {e}")
            time.sleep(5)
    else:
        print(f"Skipping chunk {current_start.date()} → {current_end.date()} due to repeated errors")
        current_start = current_end
        continue

    request_count += 1

    # Convert API data to DataFrame
    temp = pd.DataFrame(data["values"])
    temp['datetime'] = pd.to_datetime(temp['datetime'])

    # Remove overlap row unless first chunk
    if len(df) > 0:
        temp = temp[temp['datetime'] > df['datetime'].max()]

    # Append to main DataFrame
    df = pd.concat([df, temp], ignore_index=True)
    df = df.sort_values("datetime").reset_index(drop=True)

    # Save CSV and checkpoint
    df.to_csv(output_csv, index=False)
    last_downloaded_date = df['datetime'].max()
    with open(checkpoint_file, "w") as f:
        f.write(last_downloaded_date.strftime("%Y-%m-%d %H:%M:%S"))

    print(f"Saved chunk: {current_start.date()} → {current_end.date()}")

    # Move to next chunk
    current_start = current_end

print("\n🎉 Download complete!")
print(f"Total rows: {len(df)}")
print(f"Saved to: {output_csv}")


Starting fresh download from: 2021-05-01 00:00:00
Saved chunk: 2021-05-01 → 2021-05-16
Saved chunk: 2021-05-16 → 2021-05-31
Saved chunk: 2021-05-31 → 2021-06-15
Saved chunk: 2021-06-15 → 2021-06-30
Saved chunk: 2021-06-30 → 2021-07-15
Saved chunk: 2021-07-15 → 2021-07-30
Saved chunk: 2021-07-30 → 2021-08-14
Saved chunk: 2021-08-14 → 2021-08-29
⏳ Waiting 60 seconds due to API limit...
Saved chunk: 2021-08-29 → 2021-09-13
Saved chunk: 2021-09-13 → 2021-09-28
Saved chunk: 2021-09-28 → 2021-10-13
Saved chunk: 2021-10-13 → 2021-10-28
Saved chunk: 2021-10-28 → 2021-11-12
Saved chunk: 2021-11-12 → 2021-11-27
Saved chunk: 2021-11-27 → 2021-12-12
Saved chunk: 2021-12-12 → 2021-12-27
⏳ Waiting 60 seconds due to API limit...
Saved chunk: 2021-12-27 → 2022-01-11
Saved chunk: 2022-01-11 → 2022-01-26
Saved chunk: 2022-01-26 → 2022-02-10
Saved chunk: 2022-02-10 → 2022-02-25
Saved chunk: 2022-02-25 → 2022-03-12
Saved chunk: 2022-03-12 → 2022-03-27
Saved chunk: 2022-03-27 → 2022-04-11
Saved chunk: 2022

In [ ]:
df = pd.read_csv("btc_5min_dataset.csv")
df.head()

,datetime,open,high,low,close
0,2021-05-15 22:45:00,47983.92188,48146.46094,47983.92188,48146.46094
1,2021-05-15 22:50:00,48165.51953,48281.96875,48090.41016,48090.41016
2,2021-05-15 22:55:00,48084.60938,48140.80859,47928.10938,47967.66016
3,2021-05-15 23:00:00,47964.33984,47964.33984,47768.26953,47768.26953
4,2021-05-15 23:05:00,47753.12891,47753.12891,47649.51953,47697.94141


In [ ]:
df.shape

(474444, 5)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 474444 entries, 0 to 474443
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   datetime  474444 non-null  object 
 1   open      474444 non-null  float64
 2   high      474444 non-null  float64
 3   low       474444 non-null  float64
 4   close     474444 non-null  float64
dtypes: float64(4), object(1)
memory usage: 18.1+ MB


In [ ]:
df.isna().sum()

,0
datetime,0
open,0
high,0
low,0
close,0


In [ ]:
df.describe()

,open,high,low,close
count,474444.000000,474444.000000,474444.000000,474444.000000
mean,54130.840940,54184.395478,54076.108938,54130.258329
std,30357.468601,30381.047021,30333.711145,30357.855344
min,15574.690430,15587.929690,15500.339840,15579.719730
25%,28450.541993,28477.948245,28424.467770,28450.035000
50%,43970.380000,44007.695315,43927.509140,43969.979375
75%,69411.342500,69461.610000,69360.860000,69411.167500
max,126099.210000,126296.000000,125942.720000,126099.220000
